# Formulas del entorno V2V (Li et al. 2022, Sec. IV-A)

Este notebook desglosa, con LaTeX y explicacion en prosa, **todas las formulas usadas en la implementacion** del entorno (`v2v_env/params.py`, `channel.py`, `physics.py`, `reward.py`, `env.py`) construido para el nodo 3 del checklist (*"Construir el entorno de comunicacion V2V"*).

Cada seccion sigue el mismo patron: **(1)** la formula en LaTeX, **(2)** que representa cada termino, **(3)** un bloque de codigo que llama directamente a las funciones reales de `v2v_env` para verificar la formula con numeros concretos -- no son numeros inventados, son la salida real de correr el codigo.

**Alcance fijado (nodo 1 del checklist):** $M=15$ canales/CUEs, $K=5$ pares V2V (no el barrido $5$-$30$ del paper), modelo de canal simplificado (path-loss + fading basico, sin la clasificacion LOS/WLOS/NLOS de la Ec. 4 del paper).

## Notacion

Para cada simbolo: **que es**, **para que sirve / donde se usa**, y el campo del codigo que le corresponde.

### Escala del problema

- **$M$** -- numero de CUEs (usuarios celulares), que es tambien el numero de canales, porque cada CUE tiene asignado permanentemente "su" canal (el CUE $m$ siempre usa el canal $m$). Por eso $M$ no es solo "cuantos canales hay para elegir": tambien es "cuantos usuarios celulares hay que proteger de la interferencia de los V2V". Se usa para fijar el tamano del estado ($3M+1$), el tamano de la accion ($M\cdot(N_p+1)$), y como limite de todos los bucles que recorren canales/CUEs. -- `params.num_cues`
- **$K$** -- numero de pares V2V, que en este entorno son literalmente los agentes de RL (uno por par). Se usa para fijar cuantas observaciones/acciones/recompensas se generan cada paso, y determina cuantos posibles "vecinos" puede tener cada agente. -- `params.num_v2v_pairs`

### Indices

- **$m \in \{1,\dots,M\}$** -- indice que recorre los canales, y por la asignacion fija, tambien identifica al CUE dueno de ese canal. Se usa como indice para "leer la posicion $m$" de cualquier vector indexado por canal (p. ej. $h_k^t[m]$). Cuando un agente "elige el canal $m$" en la practica esta decidiendo reusar el espectro asignado al CUE $m$ -- eso es justamente lo que genera interferencia hacia ese CUE en las Sec. 5-6.
- **$k \in \{1,\dots,K\}$** -- indice que identifica a un par V2V (agente) especifico. Se usa para indexar todo lo que es "propiedad" de un agente: su posicion, su potencia elegida, su cola, su observacion y (aunque la recompensa es compartida) su contribucion al termino de confiabilidad.
- **$t$** -- indice del slot de tiempo (paso de simulacion). Se usa para marcar en que momento se evalua cada cantidad -- importante porque varias formulas mezclan el valor de este slot con el del anterior (p. ej. la cola $Q_k^t$ usa la tasa $R_k^{t-1}$, y el estado usa las selecciones de canal $\zeta^{t-1}$ de los vecinos, nunca las de este mismo slot).

### Accion (que decide cada agente)

- **$\zeta_{k,m}^t \in \{0,1\}$** -- variable binaria: "el par $k$ eligio el canal $m$ en el slot $t$". Se usa como interruptor dentro de las sumas de interferencia (Ec. 7a/7b, Sec. 5): solo suman los pares que *de verdad* estan usando ese canal ahora mismo. En el codigo no aparece como un 0/1 explicito, sino como un filtro (`channels == m`) que logra lo mismo. -- decodificado en `env._decode_action`
- **$N_p$** -- cuantos niveles de potencia *distintos de cero* puede elegir un agente (aparte de "apagado"). Se usa para fijar cuantas opciones de potencia hay dentro de cada canal y, por tanto, el tamano total del espacio de acciones $M\cdot(N_p+1)$. -- `params.num_power_levels`
- **$P_k^t$** -- potencia (en vatios) con la que el par $k$ transmite en el slot $t$; es parte de lo que el agente decide. Se usa como "cuanta senal mete" en el numerador del SINR propio de $k$, y como "cuanta interferencia mete a los demas" en el denominador del SINR de cualquier otro enlace que comparta su canal. Potencia 0 = el par no transmite ese slot. -- `physics.power_levels_watts`
- **$P_m^t$** -- potencia del CUE $m$; es un dato fijo del problema, no una decision de ningun agente. Se usa igual que $P_k^t$ pero para la senal deseada del CUE, y como fuente de interferencia hacia los pares V2V que reusan su canal. -- `params.cue_tx_power_dbm`
- **$W$** -- ancho de banda de cada canal, en Hz. Se usa como factor multiplicativo en la formula de Shannon ($R=W\log_2(1+\gamma)$): convierte "bits por uso del canal" en "bits por segundo". -- `params.channel_bandwidth_hz`

### Canal y desvanecimiento (que tan bien llega la senal)

- **$h_k^t[m]$** -- que tan bien (o mal) le llegaria la senal al par $k$ si transmitiera por el canal $m$ en el slot $t$; combina la distancia y el desvanecimiento aleatorio. Se usa multiplicado por la potencia para obtener la potencia de senal recibida -- tanto para el enlace propio de $k$ como, usando otro par de posiciones (transmisor, receptor), para calcular cuanta interferencia produce un transmisor sobre un receptor ajeno. -- `channel.large_scale_fading` x `small_scale_fading`
- **$\alpha$** (desvanecimiento de gran escala / path loss) -- la parte "lenta" y casi-deterministica del coeficiente de canal: depende solo de la distancia entre transmisor y receptor, y no cambia de un slot a otro dentro del mismo episodio (las posiciones no se mueven en esta version). Se usa para capturar cuanta senal se pierde simplemente por la distancia recorrida. -- `channel.large_scale_fading`
- **$g$** (desvanecimiento de pequena escala / fading rapido) -- la parte "rapida" y aleatoria del coeficiente de canal (efecto multipath). Se usa para simular que, aun a la misma distancia, la senal recibida fluctua de slot en slot y de canal en canal -- es lo que hace que "elegir el mejor canal" no sea una decision trivial ni estatica, sino algo que el agente tiene que aprender a manejar bajo incertidumbre. -- `channel.small_scale_fading`
- **$\sigma^2$** -- potencia del ruido de fondo (termico), siempre presente en cualquier receptor aunque no haya ningun transmisor interfiriendo. Se usa como termino base (siempre positivo) en el denominador de todo calculo de SINR, para que el SINR nunca sea infinito incluso sin interferencia. -- `physics.noise_power_watts`

### Calidad del enlace

- **$\gamma$** (SINR) -- la relacion entre la potencia de la senal deseada y todo lo que la "ensucia" (ruido + interferencia de otros). Se usa como entrada a la formula de Shannon para obtener la tasa de bits alcanzable, y es en si mismo un indicador directo de que tan limpio esta un enlace. -- `physics.sinr`
- **$R$** -- tasa de transmision en bits/segundo que un enlace logra, dado su SINR. Para el CUE se usa para compararla contra su minimo requerido $R_m^{min}$ dentro de la recompensa; para el V2V se usa para saber cuantos bits alcanzo a sacar de su cola ese slot. -- `physics.rate_bps`

### Trafico y cola (retraso del V2V)

- **$Q_k^t$** -- cuantos bits tiene acumulados sin enviar el transmisor del par $k$ al inicio del slot $t$ (su "atraso"). Se usa como parte del estado que observa el agente (para que sepa que tan atrasado va) y como entrada directa al termino de penalizacion de cola/retardo en la recompensa. -- `env._queue`
- **$\tau$** -- cuanto dura un slot, en segundos (la unidad de tiempo de la simulacion). Se usa para convertir tasas (bits/s) en cantidades de bits *por slot*, tanto para lo que llega a la cola como para lo que se logra enviar cada paso. -- `params.slot_duration_s`
- **$\lambda$** -- cuantos bits por segundo llegan, en promedio, al transmisor de cada par V2V (la demanda de trafico que genera el vehiculo). Se usa como la parte de "entrada" en la recursion de la cola, y para derivar $Q_{max}$. -- `params.v2v_arrival_rate_bps`
- **$D_{max}$** -- el retardo maximo que un paquete V2V puede tolerar antes de considerarse un fallo de calidad de servicio (100 ms, tomado de la norma 3GPP). Se usa junto con $\lambda$ para derivar $Q_{max}$ via la Ley de Little. -- `params.max_tolerable_delay_s`
- **$Q_{max}$** -- la longitud de cola maxima "tolerable", calculada a partir de $D_{max}$ y $\lambda$. Se usa como umbral de comparacion en el termino de penalizacion de cola de la recompensa: si $Q_k$ supera esto, el agente recibe el castigo plano $A$. -- `params.max_queue_length_bits`

### Confiabilidad (probabilidad de que el enlace falle)

- **$\gamma_o$** -- el umbral de SINR por debajo del cual se considera que un enlace V2V esta en "outage" (fallo momentaneo de confiabilidad). Se usa dentro de la formula del margen de confiabilidad, junto con $p_o$, para fijar que tan exigente es el requisito de calidad del enlace. -- `params.sinr_threshold_db`
- **$p_o$** -- la probabilidad maxima tolerada de que ocurra ese outage. Se usa para calcular el lado derecho de la restriccion de confiabilidad ($\gamma_o/\ln(1/(1-p_o))$): mientras mas pequeno $p_o$ (mas exigente), mas grande ese umbral y mas dificil de superar para el agente. -- `params.outage_probability_threshold`

### Recompensa

- **$U(\cdot)$** -- la funcion que convierte "que tan bien se cumple una condicion de calidad" en un numero de recompensa. Se usa igual en los tres terminos de la Ec. 22: si el margen es positivo la recompensa ES ese margen (mas margen = mas recompensa); si es negativo o cero, la recompensa cae a un castigo fijo $A$ sin importar cuanto se fallo. -- `reward.utility`
- **$A$** ($A<0$) -- el valor que se asigna cuando una condicion de QoS *no* se cumple. Se usa como "piso" de castigo, para que violar una restriccion sea consistentemente malo para la recompensa, sin importar que tan grave fue la violacion. -- `params.penalty_constant`
- **$\lambda_1,\lambda_2,\lambda_3$** -- los pesos que determinan cuanto importa cada uno de los tres objetivos (tasa de las CUEs, cola de los V2V, confiabilidad de los V2V) dentro de la recompensa total. Se usan multiplicando cada suma de terminos $U(\cdot)$ antes de sumarlas -- controlan el balance relativo entre "cuidar a las CUEs" y "cuidar a los V2V". -- `params.reward_weights`
- **$R_m^{min}$** -- la tasa minima que cada CUE necesita para considerarse bien atendido. Se usa como umbral de comparacion en el primer termino de la recompensa: si $R_m$ queda por debajo, penalizacion fija $A$; si queda por encima, recompensa proporcional al excedente. -- `params.cue_min_rate_bps`

In [1]:
import numpy as np
from v2v_env import channel, physics
from v2v_env.params import V2VEnvParams
from v2v_env.reward import utility, compute_reward
from v2v_env.env import V2VEnv

params = V2VEnvParams()  # defaults del nodo 1: M=15, K=5
print(f"M={params.num_cues}  K={params.num_v2v_pairs}  Np={params.num_power_levels}")

M=15  K=5  Np=3


## 1. Espacio de estado (Sec. IV-A.1) -- dimension $3M+1$

$$s_k^t = \Big[\, \underbrace{h_k^t[1],\dots,h_k^t[M]}_{M},\;\; \underbrace{h_{m,B}^t[1],\dots,h_{m,B}^t[M]}_{M}, \;\; \underbrace{N_k^{1,t-1},\dots,N_k^{M,t-1}}_{M},\;\; Q_k^t \,\Big]$$

Cuatro bloques, en este orden (implementado en `env._build_observations`):

1. **$h_k^t[m]\ \forall m$** -- el coeficiente de canal del *propio* enlace V2V del par $k$, en cada uno de los $M$ canales (no solo el que eligio la ultima vez): le dice al agente que tan buena es cada opcion de canal para su propio enlace.
2. **$h_{m,B}^t[m]\ \forall m$** -- el coeficiente de canal entre cada CUE y la BS: le deja al agente estimar cuanto dano le haria a cada CUE si reusara ese canal.
3. **$N_k^{m,t-1}\ \forall m$** -- cuantos vecinos (a $\le 300$m del par $k$) eligieron el canal $m$ en el slot *anterior*: unica informacion sobre otros agentes que recibe $k$, y esta retrasada un slot (nunca observa la accion actual de sus vecinos).
4. **$Q_k^t$** -- la longitud de la cola de su propio transmisor.

Nunca aparece el estado global $\mathcal{S}^t$ ni el estado privado de otro agente -- observabilidad estrictamente local, tal como exige el nodo 3 del checklist.

In [2]:
env = V2VEnv(params)
obs, infos = env.reset(seed=0)
print(f"observation_space: {env.observation_space('v2v_0')}")
print(f"dim = {env.observation_space('v2v_0').shape[0]}  ==  3M+1 = {3*params.num_cues+1}")
print()
o = obs['v2v_0']
M = params.num_cues
print(f"h_k[1..M]      (propio):  {o[0:M]}")
print(f"h_{{m,B}}[1..M]   (CUEs):    {o[M:2*M]}")
print(f"N_k[1..M]      (vecinos): {o[2*M:3*M]}  (todo cero: no hay historia en t=0)")
print(f"Q_k            (cola):    {o[-1]}")

observation_space: Box(0.0, inf, (46,), float64)
dim = 46  ==  3M+1 = 46

h_k[1..M]      (propio):  [1.16393720e-09 2.81941843e-10 1.61510994e-09 5.50004388e-11
 1.42753951e-10 1.51795808e-10 3.89925167e-10 4.77856476e-10
 2.73689597e-10 6.36059346e-11 1.02031306e-09 2.54039602e-10
 5.79966808e-11 3.93846848e-10 1.51413296e-10]
h_{m,B}[1..M]   (CUEs):    [2.29939396e-11 2.88575759e-11 6.56596237e-11 3.57270384e-10
 3.69336422e-12 1.96514545e-12 4.11908066e-12 2.32465045e-11
 4.56425763e-12 1.21357964e-11 8.39227309e-12 2.32658978e-10
 4.02664438e-12 3.21315966e-11 2.38237466e-12]
N_k[1..M]      (vecinos): [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]  (todo cero: no hay historia en t=0)
Q_k            (cola):    0.0


## 2. Espacio de accion (Sec. IV-A.2) -- dimension $M\cdot(N_p+1)$

$$a_k^t = (\zeta_{k,1},\dots,\zeta_{k,M},\; P_k^t), \qquad P_k^t \in \Big\{0,\ \tfrac{1}{N_p}P_{max}^v,\ \tfrac{2}{N_p}P_{max}^v,\ \dots,\ P_{max}^v\Big\}$$

El agente elige simultaneamente **que canal** reusar ($\zeta_{k,m}=1$ para exactamente un $m$) y **con que nivel de potencia** transmitir, discretizado en $N_p+1$ niveles igualmente espaciados entre $0$ (no transmite) y la potencia maxima $P_{max}^v$. Se codifica como un unico `Discrete(M*(N_p+1))`:

$$\text{action\_id} = m \cdot (N_p+1) + i, \qquad m = \left\lfloor \frac{\text{action\_id}}{N_p+1} \right\rfloor,\quad i = \text{action\_id} \bmod (N_p+1)$$

con $i\in\{0,\dots,N_p\}$ el indice de nivel de potencia, $P_k = \frac{i}{N_p}P_{max}^v$.

In [3]:
print(f"action_space: {env.action_space('v2v_0')}  ==  M*(Np+1) = {params.num_cues*(params.num_power_levels+1)}")
print()
levels = physics.power_levels_watts(params.v2v_max_tx_power_dbm, params.num_power_levels)
print(f"Niveles de potencia (W): {levels}")
print()
for action_id in [0, 1, params.num_power_levels, params.num_power_levels + 1]:
    ch, lvl = env._decode_action(action_id)
    print(f"action_id={action_id:>3} -> canal={ch}, nivel={lvl} (P={levels[lvl]:.4f} W)")

action_space: Discrete(60)  ==  M*(Np+1) = 60

Niveles de potencia (W): [0.         0.06650874 0.13301749 0.19952623]

action_id=  0 -> canal=0, nivel=0 (P=0.0000 W)
action_id=  1 -> canal=0, nivel=1 (P=0.0665 W)
action_id=  3 -> canal=0, nivel=3 (P=0.1995 W)
action_id=  4 -> canal=1, nivel=0 (P=0.0000 W)


## 3. Modelo de canal (Ec. 3, version simplificada de la Ec. 4)

$$h_k^t[m] = \alpha_k^t \cdot g_k^t[m]$$

El coeficiente de canal es el producto de dos factores independientes:

- **$\alpha_k^t$ (gran escala / path loss):** depende solo de la distancia, practicamente constante durante el episodio (las posiciones no se mueven en esta primera version). El paper (Ec. 4) lo separa en tres casos geometricos (LOS/WLOS/NLOS segun el carril); el nodo 1 del checklist decidio simplificar eso a **una sola ley de potencia**:

$$\alpha = \varphi_{lin}\cdot d^{-e}, \qquad \varphi_{lin} = 10^{\varphi_{dB}/10}$$

con $\varphi_{dB}$ el coeficiente de path loss (Tabla II del paper, $-68.5$ dB) y $e$ el exponente de path loss (Tabla II, $1.61$). La distancia se acota por abajo (`min_distance_m`) para que $\alpha$ no explote cuando $d\to0$.

- **$g_k^t[m]$ (pequena escala / fading rapido):** varia por slot *y* por canal ("frequency-dependent" en palabras del paper), distribucion exponencial de media unitaria:

$$g \sim \mathrm{Exp}(\text{media}=1), \qquad \mathbb{E}[g]=1$$

Esta misma pareja $(\alpha,g)$ se reutiliza para *todos* los coeficientes de interferencia de la Sec. IV-B ($h_{k,B}$, $h_{m,k}$, $h_{k',k}$) -- solo cambia que par de posiciones (transmisor, receptor) se usa para $d$.

In [4]:
rng = np.random.default_rng(0)
d = np.array([10.0, 50.0, 200.0])
alpha = channel.large_scale_fading(d, params)
print('alpha (gran escala) a 10/50/200 m:', alpha)
print('  -> decrece con la distancia, como exige un path-loss:', np.all(np.diff(alpha) < 0))
print()
g = channel.small_scale_fading(rng, size=200_000)
print(f'g (pequena escala): media empirica = {g.mean():.4f}  (teorica = 1.0)')

alpha (gran escala) a 10/50/200 m: [3.46736850e-09 2.59811454e-10 2.78831418e-11]
  -> decrece con la distancia, como exige un path-loss: True

g (pequena escala): media empirica = 0.9986  (teorica = 1.0)


## 4. Potencia de ruido

$$\sigma^2 = 10^{\,N_0[\text{dBm/Hz}]/10}\cdot 10^{-3}\cdot W$$

$N_0$ es la densidad espectral de potencia de ruido termico en dBm/Hz (Tabla II: $-174$ dBm/Hz); se convierte a W/Hz y se multiplica por el ancho de banda $W$ del canal para obtener la potencia de ruido total en esa banda.

In [5]:
sigma2 = physics.noise_power_watts(params.channel_bandwidth_hz, params.noise_psd_dbm_per_hz)
print(f"sigma^2 = {sigma2:.3e} W  (para W={params.channel_bandwidth_hz:.0e} Hz, N0={params.noise_psd_dbm_per_hz} dBm/Hz)")

sigma^2 = 3.981e-15 W  (para W=1e+06 Hz, N0=-174.0 dBm/Hz)


## 5. Interferencia (Ec. 7a, 7b)

Cuando dos enlaces reusan el mismo canal $m$, cada uno recibe interferencia del otro. Tres fuentes de interferencia aparecen en el modelo:

**Del CUE $m$ hacia el par V2V $k$ que reusa su canal:**
$$I_k^{c,t}[m] = P_m^t\, h_{m,k}^t[m]$$

**De los otros pares V2V que tambien reusan el canal $m$, hacia el par $k$:**
$$I_k^{v,t}[m] = \sum_{k'\neq k} \zeta_{k',m}^t\, P_{k'}^t\, h_{k',k}^t[m]$$

**De los pares V2V que reusan el canal $m$, hacia el CUE $m$ (interferencia "de subida" a la BS):**
$$I_m^{t} = \sum_{k\in\mathcal{K}} \zeta_{k,m}^t\, P_k^t\, h_{k,B}^t[m]$$

El indicador $\zeta_{k,m}^t$ hace que solo sumen los pares que **de verdad** eligieron el canal $m$ en este slot -- en el codigo (`env.step`) esto se implementa filtrando por `channels == canal_objetivo` en vez de multiplicar por un indicador explicito, pero es la misma suma.

In [6]:
# Ejemplo con 2 pares V2V reusando el mismo canal 0, potencia maxima
env2 = V2VEnv(V2VEnvParams(num_cues=4, num_v2v_pairs=2, num_power_levels=1))
env2.reset(seed=0)
levels2 = physics.power_levels_watts(env2.params.v2v_max_tx_power_dbm, env2.params.num_power_levels)
actions = {'v2v_0': 0 * 2 + 1, 'v2v_1': 0 * 2 + 1}  # ambos: canal 0, potencia maxima
_, rewards, _, _, infos = env2.step(actions)
print('acciones decodificadas:', {a: (info['channel'], info['power_w']) for a, info in infos.items()})
print('-> ambos en el mismo canal: cada uno le mete interferencia I_v al otro (ver seccion 7)')

acciones decodificadas: {'v2v_0': (0, 0.1995262314968879), 'v2v_1': (0, 0.1995262314968879)}
-> ambos en el mismo canal: cada uno le mete interferencia I_v al otro (ver seccion 7)


## 6. SINR (Ec. 5 y 6)

**SINR del CUE $m$:**
$$\gamma_m^t = \dfrac{P_m^t\, h_{m,B}^t[m]}{\sigma^2 + \sum_{k\in\mathcal{K}} \zeta_{k,m}^t P_k^t h_{k,B}^t[m]}$$

**SINR del par V2V $k$ (en el canal $m$ que eligio):**
$$\gamma_k^t[m] = \dfrac{P_k^t\, h_k^t[m]}{\sigma^2 + I_k^{c,t}[m] + I_k^{v,t}[m]}$$

Misma forma en ambos casos: potencia recibida de la senal deseada, dividida entre ruido termico mas toda la interferencia de co-canal. Si la potencia de transmision es 0 (par apagado), el SINR es 0 por definicion (no hay senal que interferir ni que recibir).

In [7]:
print('sinr(rx=4W, noise=1W, interferencia=1W) =', physics.sinr(4.0, 1.0, 1.0), ' (esperado 4/(1+1)=2.0)')
print('sinr(rx=0W, ...) =', physics.sinr(0.0, 1.0, 5.0), ' (potencia 0 -> SINR 0, no hay transmision)')

sinr(rx=4W, noise=1W, interferencia=1W) = 2.0  (esperado 4/(1+1)=2.0)
sinr(rx=0W, ...) = 0.0  (potencia 0 -> SINR 0, no hay transmision)


## 7. Tasa de transmision (Ec. 8, formula de Shannon)

$$R_m^t = W\cdot\log_2\!\big(1+\gamma_m^t\big) \qquad\text{(CUE)}$$
$$R_k^t = \begin{cases} W\cdot\log_2\!\big(1+\gamma_k^t[m]\big) & \text{si } P_k^t>0\\[2pt] 0 & \text{si } P_k^t=0 \end{cases} \qquad\text{(V2V, an\'aloga a la Ec. 8 para el propio enlace)}$$

Capacidad de canal ruidoso estandar. La version V2V no esta numerada por separado en el paper, pero se usa exactamente igual para calcular cuantos bits logra enviar el transmisor de $k$ en el slot -- es lo que alimenta la recursion de cola (seccion 8).

In [8]:
r = physics.rate_bps(bandwidth_hz=1.0e6, sinr_linear=3.0)
print(f"R = W*log2(1+3) = {r:.0f} bps  (esperado {1.0e6*np.log2(4):.0f} bps)")

R = W*log2(1+3) = 2000000 bps  (esperado 2000000 bps)


## 8. Dinamica de la cola (Ec. 16; $Q_{max}$ via Ec. 17-18, Ley de Little)

$$Q_k^t = \max\!\Big(0,\; Q_k^{t-1} + \tau\lambda - \tau R_k^{t-1}\Big)$$

Cada slot llegan $\tau\lambda$ bits nuevos (tasa de trafico $\lambda$ por la duracion del slot $\tau$) y salen $\tau R_k^{t-1}$ bits (lo que el transmisor logro enviar el slot anterior, a su tasa $R_k^{t-1}$). La cola nunca baja de 0. **Importante de causalidad:** $Q_k^t$ usa la tasa del slot *anterior* -- por eso $Q_k^t$ ya es parte del estado $s^t$ observado *antes* de elegir la accion $a^t$, y es ese mismo valor el que entra en el termino de recompensa de la Ec. 22 (la recompensa depende de $(s^t,a^t)$, no de $Q_k^{t+1}$).

$Q_{max}$ (cota usada en la recompensa, no en esta recursion) se deriva de la Ley de Little a partir del retardo maximo tolerable:
$$Q_{max} = \lambda \cdot D_{max}$$

In [9]:
q1 = physics.update_queue(prev_queue=0.0, arrival_rate_bps=1000.0, slot_duration_s=0.01, rate_bps=0.0)
print(f"cola tras 1 slot sin transmitir nada: {q1} bits  (esperado 1000*0.01 = 10.0)")

q2 = physics.update_queue(prev_queue=0.0, arrival_rate_bps=0.0, slot_duration_s=0.01, rate_bps=1000.0)
print(f"cola no baja de 0 aunque se transmita sin llegadas: {q2} bits")

print(f"Q_max = lambda * D_max = {params.v2v_arrival_rate_bps} * {params.max_tolerable_delay_s} = {params.max_queue_length_bits} bits")

cola tras 1 slot sin transmitir nada: 10.0 bits  (esperado 1000*0.01 = 10.0)
cola no baja de 0 aunque se transmita sin llegadas: 0.0 bits
Q_max = lambda * D_max = 1000000.0 * 0.1 = 100000.0 bits


## 9. Margen de confiabilidad (restriccion de outage, Ec. 15)

La confiabilidad de un enlace V2V se mide por su probabilidad de outage (SINR instantaneo cae bajo un umbral $\gamma_o$). El paper deriva, via la desigualdad de Markov y el hecho de que $g$ tiene media 1, una restriccion determinista equivalente sobre el SINR *esperado* (usa $\alpha$ solo, sin el fading rapido $g$):

$$\underbrace{\dfrac{P_k^t\,\alpha_k^t}{\sigma^2 + I_k^{c,t}[m] + \sum_{k'\neq k}\zeta_{k',m}^t P_{k'}^t \alpha_{k',k}^t}}_{\text{lado izquierdo}} \;\ge\; \underbrace{\dfrac{\gamma_o}{\ln\!\big(\tfrac{1}{1-p_o}\big)}}_{\text{lado derecho}}$$

El **margen** que se usa como tercer termino de la recompensa (Ec. 22) es simplemente lado-izquierdo menos lado-derecho:

$$\text{margen}_k = \dfrac{P_k^t\,\alpha_k^t}{\sigma^2 + I_k^{c,t}[m] + \sum_{k'\neq k}\zeta_{k',m}^t P_{k'}^t \alpha_{k',k}^t} - \dfrac{\gamma_o}{\ln\!\big(\tfrac{1}{1-p_o}\big)}$$

Margen positivo = la restriccion de confiabilidad se cumple (con ese margen de sobra); margen negativo = se viola.

In [10]:
m_ok = physics.reliability_margin(tx_power_w=1.0, alpha=1.0, noise_w=0.01, interference_w=0.0,
                                    sinr_threshold_db=0.0, outage_probability_threshold=0.5)
print(f"margen con senal fuerte y poco ruido: {m_ok:.2f}  (positivo -> confiable)")

m_bad = physics.reliability_margin(tx_power_w=1e-6, alpha=1e-6, noise_w=1.0, interference_w=1.0,
                                     sinr_threshold_db=10.0, outage_probability_threshold=0.1)
print(f"margen con senal debil y mucho ruido:  {m_bad:.2f}  (negativo -> viola confiabilidad)")

margen con senal fuerte y poco ruido: 98.56  (positivo -> confiable)
margen con senal debil y mucho ruido:  -94.91  (negativo -> viola confiabilidad)


## 10. Funcion de utilidad y recompensa compartida (Ec. 22 y 23)

$$U(x) = \begin{cases} x, & \text{si } x>0 \\ A, & \text{si } x\le 0\end{cases}, \qquad A<0$$

$U$ es el mismo patron en los tres terminos de la recompensa: si la restriccion de QoS correspondiente se cumple ($x>0$), la recompensa crece proporcional al margen; si se viola ($x\le0$), cae a un castigo plano $A$ *independiente de que tan mal* se violo.

$$\mathcal{R}^t = \lambda_1\sum_{m\in\mathcal{M}} U\big(R_m^t - R_m^{min}\big) \;+\; \lambda_2\sum_{k\in\mathcal{K}} U\big(Q_{max} - Q_k^t\big) \;+\; \lambda_3\sum_{k\in\mathcal{K}} U(\text{margen}_k)$$

| Termino | Representa | Se cumple cuando |
|---|---|---|
| $\lambda_1\sum_m U(R_m^t-R_m^{min})$ | tasa de suma de las CUEs, con requisito minimo de QoS | cada CUE alcanza al menos su tasa minima $R_m^{min}$ |
| $\lambda_2\sum_k U(Q_{max}-Q_k^t)$ | penalizacion de cola/retardo del V2V | la cola de $k$ no excede $Q_{max}$ |
| $\lambda_3\sum_k U(\text{margen}_k)$ | confiabilidad/interferencia del V2V | el SINR esperado de $k$ cumple el umbral de outage |

**Desviacion deliberada de la Ec. 22 tal como aparece impresa en el paper (importante, ver README/`reward.py`):** el termino de cola en el PDF se lee literalmente $U(Q_k^t - Q_{max})$, no $U(Q_{max}-Q_k^t)$. Con esa forma literal, una cola que *supera* $Q_{max}$ suma un valor cada vez *mas positivo* mientras mas se desborda -- exactamente al reves de una "penalizacion de cola", y en contradiccion con la restriccion (19e) del propio paper ($Q_k^t\le Q_{max}$) y con como el nodo 3 del checklist describe este termino. Los otros dos terminos SI siguen el patron "$U(\text{lado que debe ser grande} - \text{umbral})$" consistente con sus restricciones ($\ge$). Para que el termino de cola siga el mismo patron dado que su restriccion es $\le$, tiene que ser $U(Q_{max}-Q_k^t)$ -- asi quedo implementado aqui. Vale la pena confirmarlo contra la copia de Nicolas o el PDF original antes de citarlo en la tesis.

In [11]:
print('U(5, A=-10) =', utility(5.0, penalty=-10.0), ' (x>0 -> recompensa = margen)')
print('U(-3, A=-10) =', utility(-3.0, penalty=-10.0), ' (x<=0 -> castigo plano A)')
print()

r = compute_reward(
    cue_rates=np.array([2.0e6]), cue_min_rate_bps=1.0e6,
    v2v_queues=np.array([10.0]), max_queue_length_bits=100.0,
    reliability_margins=np.array([1.0]),
    weights=(0.3, 0.5, 0.2), penalty=-10.0,
)
expected = 0.3 * (2.0e6 - 1.0e6) + 0.5 * (100.0 - 10.0) + 0.2 * 1.0
print(f'compute_reward(...) = {r}')
print(f'lambda1*(R-Rmin) + lambda2*(Qmax-Q) + lambda3*margen = {expected}')

U(5, A=-10) = 5.0  (x>0 -> recompensa = margen)
U(-3, A=-10) = -10.0  (x<=0 -> castigo plano A)

compute_reward(...) = 300045.2
lambda1*(R-Rmin) + lambda2*(Qmax-Q) + lambda3*margen = 300045.2


## 11. Conteo de vecinos (detalle del bloque 3 del estado)

$$\mathrm{neighbors}(k) = \{k' : \lVert x_k - x_{k'} \rVert \le 300\text{m}\}, \qquad N_k^{m,t-1} = \sum_{k'\in \mathrm{neighbors}(k)} \mathbf{1}\big[\zeta_{k',m}^{t-1}=1\big]$$

$x_k$ es la posicion del transmisor del par $k$. $N_k^{m,t-1}$ cuenta, entre los vecinos de $k$ a 300m o menos, cuantos eligieron el canal $m$ en el slot anterior -- la unica senal indirecta que $k$ recibe sobre las acciones de otros agentes.

In [12]:
tx = np.array([[0.0, 0.0], [1.0, 0.0], [1000.0, 0.0]])
mask = channel.neighbor_mask(tx, radius_m=300.0)
print('mascara de vecinos (True = vecino):')
print(mask)
print()
prev_channels = np.array([0, 0, 2])
counts = channel.neighbor_channel_counts(mask, prev_channels, num_channels=3)
print('N_k^{m,t-1} por agente (filas) y canal (columnas):')
print(counts)

mascara de vecinos (True = vecino):
[[False  True False]
 [ True False False]
 [False False False]]

N_k^{m,t-1} por agente (filas) y canal (columnas):
[[1 0 0]
 [1 0 0]
 [0 0 0]]


## 12. Parametros libres (no tabulados por el paper)

Sec. IV-A/V del paper no da un valor numerico para estos; se dejaron como parametros configurables (`V2VEnvParams`), documentados en el codigo:

| Parametro | Simbolo | Valor por defecto | Rol |
|---|---|---|---|
| `penalty_constant` | $A$ | $-10$ | castigo plano al violar cualquier restriccion de QoS |
| `outage_probability_threshold` | $p_o$ | $0.1$ | umbral de probabilidad de outage aceptable |
| `cue_min_rate_bps` | $R_m^{min}$ | $1$ Mbps | tasa minima requerida por CUE |
| `v2v_arrival_rate_bps` | $\lambda$ | $1$ Mbps | tasa de llegada de trafico por par V2V |
| `slot_duration_s` | $\tau$ | $1$ ms | duracion de un slot |
| `cell_radius_m`, rango de distancia V2V | -- | 500 m, 10-50 m | geometria de posiciones (no dada por el paper al simplificar la Ec. 4) |

Estos son exactamente los que conviene revisar/ajustar antes de pasar al nodo 4 (agentes D3QN) -- en particular $A$, dado el hallazgo de la siguiente seccion.

## 13. Nota de verificacion (subtarea 4 del checklist): desbalance de escala

Al "probar el entorno con acciones aleatorias y verificar que la recompensa se comporta razonablemente" (subtarea 4), se compararon dos acciones conjuntas con la misma semilla: los 5 pares en canales distintos vs. los 5 colisionando en el mismo canal, ambos a maxima potencia. El termino de confiabilidad reacciona correctamente (positivo -> muy negativo con la colision), pero la recompensa **total** sube con la colision, porque el termino de tasa CUE (escala ~$10^6$-$10^7$ bps, sumado sobre $M=15$ CUEs) domina por varios ordenes de magnitud a los terminos de cola y confiabilidad (escala ~$10$-$500$). Concentrar la interferencia en un solo canal libera a las otras $M-1$ CUEs, y esa ganancia agregada aplasta el castigo de confiabilidad. No es un bug: es una propiedad emergente de sumar bps crudos sobre $M \gg K$ canales con estos pesos $(\lambda_1,\lambda_2,\lambda_3)$. Antes de entrenar (nodo 4) conviene reescalar (p. ej. tasas en Mbps, o un $|A|$ mayor) para que los tres terminos pesen de forma comparable.

In [13]:
import v2v_env.reward as reward_mod

captured = {}
orig_compute = reward_mod.compute_reward
def _spy(**kwargs):
    captured['kwargs'] = kwargs
    return orig_compute(**kwargs)
import v2v_env.env as env_mod
env_mod.compute_reward = _spy

real_params = V2VEnvParams(num_cues=15, num_v2v_pairs=5, num_power_levels=3, max_steps=1)
agents = [f'v2v_{i}' for i in range(5)]

def run_scenario(actions):
    e = V2VEnv(real_params)
    e.reset(seed=1)
    e.step(actions)
    kw = captured['kwargs']
    l1, l2, l3 = kw['weights']
    cue = l1 * sum(utility(r - kw['cue_min_rate_bps'], kw['penalty']) for r in kw['cue_rates'])
    que = l2 * sum(utility(kw['max_queue_length_bits'] - q, kw['penalty']) for q in kw['v2v_queues'])
    rel = l3 * sum(utility(m, kw['penalty']) for m in kw['reliability_margins'])
    return cue, que, rel

distinct = {a: i * 4 + 3 for i, a in enumerate(agents)}   # canales 0..4, potencia maxima
collide = {a: 0 * 4 + 3 for a in agents}                   # todos en canal 0, potencia maxima

for label, actions in [('canales distintos', distinct), ('todos colisionan', collide)]:
    cue, que, rel = run_scenario(actions)
    print(f"{label:>18}: cue={cue:>14,.1f}  cola={que:>10,.1f}  confiabilidad={rel:>8,.1f}  total={cue+que+rel:>14,.1f}")

 canales distintos: cue=  25,844,161.7  cola= 250,000.0  confiabilidad=   146.8  total=  26,094,308.5
  todos colisionan: cue=  34,029,030.6  cola= 250,000.0  confiabilidad=   -10.0  total=  34,279,020.6
